# From Scrape Payloads to Parquet and DuckDB

This notebook is an intro for someone comfortable with CSV files, pandas, and basic SQL.
It shows how data moves from `pystocks/fundamentals.py` into the parquet stores produced by `pystocks/fundamentals_store.py`, and how DuckDB views sit on top.

## Mental Model (CSV -> Parquet -> DuckDB)

- `fundamentals.py` fetches endpoint JSON payloads per `conid`.
- `fundamentals_store.py` persists each payload in multiple forms:
  - Raw JSON blob (compressed, content-addressed storage)
  - Slim endpoint snapshot parquet row (lineage/audit)
  - Normalized feature parquet rows (long table format)
  - Dedicated series parquet stores (price, sentiment, ownership trade log, dividends events)
- DuckDB reads those parquet files through views so you can use SQL without manually loading files in pandas.

In [5]:
from pathlib import Path
import json
import sqlite3

import duckdb
import pandas as pd
import zstandard as zstd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [6]:
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'

paths = {
    'events_db': DATA / 'fundamentals' / 'events.db',
    'duckdb': DATA / 'fundamentals' / 'fundamentals.duckdb',
    'blobs_dir': DATA / 'fundamentals' / 'blobs',
    'endpoint_parquet_dir': DATA / 'fundamentals' / 'parquet',
    'factor_dir': DATA / 'factors' / 'ibkr_factor_features',
    'price_dir': DATA / 'prices' / 'ibkr_mf_performance_chart',
    'clean_price_dir': DATA / 'prices' / 'ibkr_mf_performance_chart_clean',
    'sentiment_dir': DATA / 'sentiment' / 'ibkr_sma_search',
    'ownership_dir': DATA / 'ownership' / 'ibkr_ownership_trade_log',
    'dividends_dir': DATA / 'dividends' / 'ibkr_dividends_events',
    'research_dir': DATA / 'research',
}

pd.DataFrame([{'artifact': k, 'path': str(v), 'exists': v.exists()} for k, v in paths.items()])

,artifact,path,exists
0,events_db,/home/alex/Documents/pystocks/data/fundamentals/events.db,True
1,duckdb,/home/alex/Documents/pystocks/data/fundamentals/fundamentals.duckdb,True
2,blobs_dir,/home/alex/Documents/pystocks/data/fundamentals/blobs,True
3,endpoint_parquet_dir,/home/alex/Documents/pystocks/data/fundamentals/parquet,True
4,factor_dir,/home/alex/Documents/pystocks/data/factors/ibkr_factor_features,True
5,price_dir,/home/alex/Documents/pystocks/data/prices/ibkr_mf_performance_chart,True
6,clean_price_dir,/home/alex/Documents/pystocks/data/prices/ibkr_mf_performance_chart_clean,True
7,sentiment_dir,/home/alex/Documents/pystocks/data/sentiment/ibkr_sma_search,True
8,ownership_dir,/home/alex/Documents/pystocks/data/ownership/ibkr_ownership_trade_log,True
9,dividends_dir,/home/alex/Documents/pystocks/data/dividends/ibkr_dividends_events,True


## 1) What `fundamentals.py` Produced in the Most Recent Run

In [10]:
telemetry_path = paths['research_dir'] / 'fundamentals_run_telemetry_latest.json'
telemetry = json.loads(telemetry_path.read_text())

run_stats = pd.DataFrame([telemetry['run_stats']])
endpoint_summary = pd.DataFrame(telemetry['endpoint_summary']).sort_values('endpoint')

display(pd.DataFrame([{
    'run_started_at': telemetry['run_started_at'],
    'run_finished_at': telemetry['run_finished_at'],
    'endpoint_families': len(endpoint_summary),
}]))
# display(telemetry)
display(run_stats)
endpoint_summary

,run_started_at,run_finished_at,endpoint_families
0,2026-02-24T16:08:45.472912+00:00,2026-02-24T16:20:14.546428+00:00,12


,total_targeted_conids,processed_conids,saved_snapshots,inserted_events,duplicate_events,factor_rows_written,series_rows_written,auth_retries,aborted
0,100,100,100,854,0,31717,357252,1,False


,endpoint,call_count,useful_payload_count,useful_payload_rate,status_codes
0,dividends,89,89,1.000000,{'200': 89}
1,impact/esg,89,0,0.000000,{'400': 89}
2,landing,100,100,1.000000,{'200': 100}
3,mf_holdings,89,89,1.000000,{'200': 89}
4,mf_lip_ratings,89,85,0.955056,{'200': 89}
5,mf_performance,109,84,0.770642,{'200': 109}
6,mf_performance_chart,89,89,1.000000,{'200': 89}
7,mf_profile_and_fees,89,89,1.000000,{'200': 89}
8,mf_ratios_fundamentals,89,48,0.539326,{'200': 89}
9,mstar/fund/detail,89,85,0.955056,"{'200': 85, '404': 4}"


## 2) The Manifest Database (`events.db`)

Think of this as the transaction log for ingested payloads.
`blobs` stores unique raw payload files, and `endpoint_events` stores each ingested endpoint event with lineage metadata.

In [11]:
with sqlite3.connect(paths['events_db']) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
    counts = pd.read_sql_query("""
        SELECT 'blobs' AS table_name, COUNT(*) AS n_rows FROM blobs
        UNION ALL
        SELECT 'endpoint_events' AS table_name, COUNT(*) AS n_rows FROM endpoint_events
    """, conn)
    by_endpoint = pd.read_sql_query("""
        SELECT endpoint, COUNT(*) AS n_events, MIN(effective_at) AS min_effective_at, MAX(effective_at) AS max_effective_at
        FROM endpoint_events
        GROUP BY endpoint
        ORDER BY endpoint
    """, conn)

display(tables)
display(counts)
by_endpoint

,name
0,blobs
1,endpoint_events
2,sqlite_sequence


,table_name,n_rows
0,blobs,804
1,endpoint_events,854


,endpoint,n_events,min_effective_at,max_effective_at
0,dividends,89,2026-01-30,2026-02-10
1,holdings,89,2024-04-30,2026-01-31
2,landing,100,2026-01-30,2026-02-24
3,lipper_ratings,85,2016-10-31,2026-01-31
4,morningstar,85,2025-05-31,2026-02-11
5,ownership,14,2026-01-31,2026-01-31
6,performance,84,2026-01-30,2026-02-10
7,price_chart,89,2026-01-30,2026-02-10
8,profile_and_fees,89,2026-01-30,2026-01-30
9,ratios,48,2026-01-31,2026-01-31


## 3) Inspect One Raw Payload Blob

`fundamentals_store` compresses canonical JSON payloads with zstd and stores them by SHA256 hash.
Below, we pick one recent event and inspect its raw JSON payload keys.

In [15]:
con = duckdb.connect(str(paths['duckdb']), read_only=True)
sample_event = con.execute("""
    SELECT
        event_id,
        conid,
        endpoint,
        endpoint_slug,
        observed_at,
        effective_at,
        payload_hash,
        blob_path
    FROM endpoint_events_all
    ORDER BY TRY_CAST(observed_at AS TIMESTAMP) DESC, event_id DESC
    LIMIT 50
""").fetchdf().iloc[-1].to_dict()
con.close()

sample_event

{'event_id': 805,
 'conid': '104158269',
 'endpoint': 'profile_and_fees',
 'endpoint_slug': 'profile_and_fees',
 'observed_at': '2026-02-24T17:19:30.106431',
 'effective_at': '2026-01-30',
 'payload_hash': '25ef1532303a8cfd4ed8dfe2a0d25d6fcf15a2cbc7113e824c198a1c7079eaf0',
 'blob_path': '/home/alex/Documents/pystocks/data/fundamentals/blobs/25/ef/25ef1532303a8cfd4ed8dfe2a0d25d6fcf15a2cbc7113e824c198a1c7079eaf0.json.zst'}

In [18]:
blob_path = Path(sample_event['blob_path'])
dctx = zstd.ZstdDecompressor()
payload = json.loads(dctx.decompress(blob_path.read_bytes()).decode('utf-8'))

top_keys = list(payload.keys()) if isinstance(payload, dict) else []

display(pd.DataFrame([{
    'endpoint': sample_event['endpoint'],
    'blob_file': str(blob_path),
    'payload_type': type(payload).__name__,
    'top_level_keys': len(top_keys),
}]))

payload

,endpoint,blob_file,payload_type,top_level_keys
0,profile_and_fees,/home/alex/Documents/pystocks/data/fundamentals/blobs/25/ef/25ef1532303a8cfd4ed8dfe2a0d25d6fcf15a2cbc7113e824c198a1c7079eaf0.json.zst,dict,7


{'expenses_allocation': [{'name': 'Management Expenses',
   'ratio': 0.9976380238896462,
   'value': '99.76%'},
  {'name': 'Non-Management Expenses',
   'ratio': 0.0023619761103538675,
   'value': '0.24%'}],
 'fund_and_profile': [{'name': 'Fund Management Company',
   'name_tag': 'Fund_Management_Company',
   'value': 'SSgA Funds Management Inc'},
  {'name': 'Distribution Details',
   'name_tag': 'Distribution_Details',
   'value': 'Paid',
   'value_tag': 'paid_tag'},
  {'name': 'Fiscal Date', 'name_tag': 'Fiscal_Date', 'value': '6/30'},
  {'name': 'Launch Opening Price',
   'name_tag': 'Launch_Opening_Price',
   'value': '2012-03-14'},
  {'name': 'Geographical Focus',
   'name_tag': 'Geographical_Focus',
   'value': 'United States of America'},
  {'name': 'Asset Type', 'name_tag': 'Asset_Type', 'value': 'Bond'},
  {'name': 'Fund Manager Benchmark',
   'name_tag': 'Fund_Manager_Benchmark',
   'value': 'Index is not available on Lipper Database'},
  {'name': 'Management Approach',
   'n

## 4) Endpoint Snapshot Parquet (Audit Row)

For each endpoint event, `fundamentals_store` writes one slim parquet row under:
`data/fundamentals/parquet/endpoint=<slug>/year=<YYYY>/month=<MM>/...parquet`

This row keeps lineage fields and a small snapshot profile (not the full raw JSON).

In [ ]:
effective_at = str(sample_event['effective_at'])
year, month, _ = effective_at.split('-')
snapshot_path = (
    paths['endpoint_parquet_dir']
    / f"endpoint={sample_event['endpoint_slug']}"
    / f"year={year}"
    / f"month={month}"
    / f"{effective_at}_{sample_event['conid']}_{str(sample_event['payload_hash'])[:12]}_{int(sample_event['event_id'])}.parquet"
)

snapshot_df = pd.read_parquet(snapshot_path)
display(pd.DataFrame([{'snapshot_path': str(snapshot_path), 'rows': len(snapshot_df), 'cols': len(snapshot_df.columns)}]))
snapshot_df.head(1)

## 5) Factor Feature Parquet (Normalized Long Format)

Endpoints like `holdings`, `ratios`, `performance`, `dividends`, etc., are converted to normalized feature rows:
`feature_name`, `feature_group`, `feature_value`, plus lineage fields.

This is similar to a tidy CSV table and works well for SQL + pandas analysis.

In [ ]:
con = duckdb.connect(str(paths['duckdb']), read_only=True)
sample_factor = con.execute("""
    SELECT
        endpoint_event_id,
        endpoint,
        conid,
        effective_at,
        payload_hash,
        COUNT(*) AS n_features
    FROM factor_features_all
    GROUP BY 1,2,3,4,5
    ORDER BY n_features DESC, endpoint_event_id DESC
    LIMIT 1
""").fetchdf().iloc[0].to_dict()
con.close()

sample_factor

In [ ]:
factor_path = (
    paths['factor_dir']
    / f"endpoint={sample_factor['endpoint']}"
    / f"conid={sample_factor['conid']}"
    / f"{sample_factor['effective_at']}_{str(sample_factor['payload_hash'])[:12]}_{int(sample_factor['endpoint_event_id'])}.parquet"
)

factor_df = pd.read_parquet(factor_path)
display(pd.DataFrame([{'factor_path': str(factor_path), 'rows': len(factor_df), 'cols': len(factor_df.columns)}]))
factor_df[['endpoint', 'conid', 'feature_group', 'feature_name', 'feature_value']].head(15)

## 6) Series Parquet Stores (Price Example)

Time-series endpoints are stored separately by `conid`, for example:
`data/prices/ibkr_mf_performance_chart/conid=<conid>/series.parquet`

In [ ]:
price_file = sorted(paths['price_dir'].glob('conid=*/*.parquet'))[0]
price_df = pd.read_parquet(price_file)

display(pd.DataFrame([{'price_file': str(price_file), 'rows': len(price_df), 'cols': len(price_df.columns)}]))
price_df.head(10)

## 7) DuckDB: SQL on Top of Parquet

DuckDB views in `fundamentals.duckdb` let you query many parquet files as if they were normal SQL tables.

You can think of this as:
- CSV world: loop files + `pd.read_csv` + concat
- DuckDB world: one SQL query over a view or `read_parquet(...)`

In [ ]:
con = duckdb.connect(str(paths['duckdb']), read_only=True)
endpoint_catalog = con.execute("SELECT * FROM endpoint_catalog ORDER BY endpoint").fetchdf()
factor_catalog = con.execute("SELECT * FROM factor_features_catalog ORDER BY n_rows DESC LIMIT 20").fetchdf()
con.close()

display(endpoint_catalog)
factor_catalog

In [ ]:
con = duckdb.connect(str(paths['duckdb']), read_only=True)
from_view = con.execute("SELECT conid, COUNT(*) AS n_rows FROM price_chart_series_all GROUP BY conid ORDER BY n_rows DESC LIMIT 5").fetchdf()
from_glob = con.execute("""
    SELECT conid, COUNT(*) AS n_rows
    FROM read_parquet('data/prices/ibkr_mf_performance_chart/conid=*/*.parquet', union_by_name=true)
    GROUP BY conid
    ORDER BY n_rows DESC
    LIMIT 5
""").fetchdf()
con.close()

display(from_view)
from_glob

## 8) Where `factor_panel_long_daily` Comes From

`factor_panel_long_daily` is an as-of panel. For each `(conid, trade_date)`, it pulls the most recent feature value whose `effective_at <= trade_date`, then unions in price-derived features.

In [ ]:
con = duckdb.connect(str(paths['duckdb']), read_only=True)
sample_conid = con.execute("SELECT conid FROM price_quality_catalog WHERE eligible = TRUE LIMIT 1").fetchone()[0]
panel_slice = con.execute("""
    SELECT trade_date, feature_group, feature_name, feature_value
    FROM factor_panel_long_daily
    WHERE conid = ?
      AND trade_date >= DATE '2026-02-01'
    ORDER BY trade_date DESC, feature_group, feature_name
    LIMIT 40
""", [sample_conid]).fetchdf()
con.close()

print('sample_conid:', sample_conid)
panel_slice

## 9) Tail Outputs for Analysis

After `preprocess_prices` and `run_analysis`, check these files:
- `data/research/price_quality_report_latest.json`
- `data/research/price_quality_catalog.parquet`
- `data/research/factor_returns_daily_latest.parquet`
- `data/research/asset_factor_betas_latest.parquet`
- `data/research/analysis_summary_latest.json`

In [ ]:
price_quality = json.loads((paths['research_dir'] / 'price_quality_report_latest.json').read_text())
analysis_summary = json.loads((paths['research_dir'] / 'analysis_summary_latest.json').read_text())

display(pd.DataFrame([price_quality['summary']]))
display(pd.DataFrame([analysis_summary]))

factor_returns = pd.read_parquet(paths['research_dir'] / 'factor_returns_daily_latest.parquet')
betas = pd.read_parquet(paths['research_dir'] / 'asset_factor_betas_latest.parquet')

display(factor_returns.head())
betas.head()

## Suggested Next Steps

1. Change the endpoint filter in the manifest query and inspect a different payload family (`ownership`, `dividends`, `sentiment_search`).
2. Pick one `conid` and trace it across:
   - raw blob JSON
   - endpoint snapshot parquet
   - factor rows
   - price/sentiment/ownership/dividends series
3. Run `./venv/bin/python -m pystocks.cli scrape_fundamentals --limit 5 --verbose`, then rerun this notebook to see what changed.